# Trademarkia Semantic Search - Analysis
Exploring K selection, Fuzzy topologies, and adaptive thresholds.

In [ ]:
import sys
from pathlib import Path
import pickle

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import umap

# Add project root to sys.path
project_root = Path("..").resolve()
sys.path.append(str(project_root))

from src.config import settings
from src.vector_store import VectorStore

# Plotting config
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

In [ ]:
# 1. Load Data
vector_store = VectorStore(
    db_dir=settings.db_dir,
    space=settings.hnsw_space,
    ef_construction=settings.hnsw_ef_construction,
    m=settings.hnsw_m
)

doc_ids, X = vector_store.get_all_doc_embeddings()
print(f"Loaded {len(X)} document embeddings.")

# 2. Load Models
with open(settings.artifacts_dir / "fcm_model.pkl", 'rb') as f:
    fcm_model = pickle.load(f)
    
with open(settings.artifacts_dir / "cluster_thresholds.pkl", 'rb') as f:
    thresholds = pickle.load(f)
    
print(f"Loaded FCM model with K={fcm_model.K}")

## 1. UMAP Projection of Fuzzy Clusters

In [ ]:
# Compute UMAP reduction
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
embedding_2d = reducer.fit_transform(X)

# Extrat crisp labels for coloring
labels = np.argmax(fcm_model.U, axis=1)

# Plot
plt.figure(figsize=(12, 8))
scatter = plt.scatter(embedding_2d[:, 0], embedding_2d[:, 1], c=labels, cmap='tab20', s=5, alpha=0.6)
plt.title(f"UMAP Projection of 20 Newsgroups (K={fcm_model.K})")
plt.colorbar(scatter, label="Cluster ID")
plt.show()

## 2. Cross-Membership Heatmap

In [ ]:
# Calculate cluster co-occurrence based on soft memberships
co_membership = fcm_model.U.T @ fcm_model.U
# Normalize row-wise
co_membership = co_membership / np.diag(co_membership)[:, None]

plt.figure(figsize=(10, 8))
sns.heatmap(co_membership, annot=False, cmap="YlGnBu", xticklabels=True, yticklabels=True)
plt.title("Fuzzy Cluster Cross-Membership Probability")
plt.xlabel("Cluster ID")
plt.ylabel("Cluster ID")
plt.show()

## 3. Theta Exploration

In [ ]:
theta_values = [0.70, 0.80, 0.85, 0.92, 0.98]
print(f"Adaptive thresholds computed by model: {thresholds}")